# 01 PTQ/QAT W8A8 Quantization for SmolVLM Monarch

This notebook produces and validates a W8A8 quantized checkpoint from the stage-00 Monarch SmolVLM checkpoint. It does not benchmark or claim Raspberry Pi 5 speedup.

In [ ]:
# Section 0 — Environment and inputs
import os, socket, sys, json, gc, time, copy, math
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw
from safetensors.torch import load_file
import transformers
from transformers import AutoProcessor

from monarch_paper import MonarchRectangular, MonarchSquare
from quant_w8a8 import (
    QuantLinearW8A8, QuantMonarchW8A8,
    audit_quantizable_modules, family_for_name, get_module, set_module,
    quantize_model_w8a8, start_calibration, freeze_calibration,
    manifest_summary, quant_weight_errors, iter_quant_wrappers, freeze_qat_model_w8a8,
)

NOTEBOOK_NAME = "01_ptq_qat.ipynb"
STAGE00_NOTEBOOK = Path("00_smolvlm_monarch.ipynb")
WORKSPACE = Path.cwd()
CKPT_DIR = WORKSPACE / "monarch_checkpoints"
MONARCH_CKPT = CKPT_DIR / "final"
MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
IMAGE_PATH = WORKSPACE / "Cat03.jpg"
GPU_ID = int(os.environ.get("GPU_ID", "1"))
if torch.cuda.is_available():
    torch.cuda.set_device(GPU_ID)
DEVICE = f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print("notebook:", NOTEBOOK_NAME)
print("stage00 notebook found:", STAGE00_NOTEBOOK.exists(), STAGE00_NOTEBOOK)
print("hostname:", socket.gethostname())
print("pwd:", WORKSPACE)
print("python:", sys.executable)
print("python version:", sys.version.split()[0])
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(GPU_ID))
    print("cuda memory GB:", round(torch.cuda.get_device_properties(GPU_ID).total_memory / 1e9, 2))
print("monarch checkpoint:", MONARCH_CKPT, "exists=", MONARCH_CKPT.exists())
assert MONARCH_CKPT.exists(), "Expected monarch_checkpoints/final from stage 00"

_tv = tuple(int(x) for x in transformers.__version__.split(".")[:2])
if _tv >= (5, 0):
    from transformers import AutoModelForImageTextToText as ModelCls
else:
    from transformers import AutoModelForVision2Seq as ModelCls


In [ ]:
# Section 0b — Load dense original and rebuild Monarch checkpoint

def replace_monarch_modules_from_state(model, state):
    replaced = []
    prefixes = sorted(k[:-5] for k in state if k.endswith(".left"))
    for prefix in prefixes:
        left = state[prefix + ".left"]
        right = state[prefix + ".right"]
        g, out_block, g2 = left.shape
        rg, rg2, in_block = right.shape
        assert g == g2 == rg == rg2, (prefix, left.shape, right.shape)
        out_features = g * out_block
        in_features = g * in_block
        bias_key = prefix + ".bias"
        mod = MonarchRectangular(in_features, out_features, nblocks=g, bias=bias_key in state)
        set_module(model, prefix, mod)
        replaced.append({"name": prefix, "in": in_features, "out": out_features, "nblocks": g})
    return replaced

print("Loading processor from", MONARCH_CKPT)
processor = AutoProcessor.from_pretrained(str(MONARCH_CKPT))

print("Loading original dense model")
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
original_dense = ModelCls.from_pretrained(MODEL_ID, torch_dtype=DTYPE, _attn_implementation="eager").eval()

print("Loading base model for Monarch checkpoint")
monarch_fp = ModelCls.from_pretrained(MODEL_ID, torch_dtype=DTYPE, _attn_implementation="eager")
state = load_file(str(MONARCH_CKPT / "model.safetensors"), device="cpu")
monarch_replaced = replace_monarch_modules_from_state(monarch_fp, state)
missing, unexpected = monarch_fp.load_state_dict(state, strict=False)
print("Monarch modules rebuilt:", len(monarch_replaced))
print("missing keys:", len(missing), "unexpected keys:", len(unexpected))
if missing[:5]: print("missing sample:", missing[:5])
if unexpected[:5]: print("unexpected sample:", unexpected[:5])
monarch_fp = monarch_fp.to(dtype=DTYPE).eval()
print("models ready on CPU; move to GPU only for calibration/evaluation")


In [ ]:
# Section 1 — Audit quantization scope

audit_rows = audit_quantizable_modules(monarch_fp)
summary = []
acc = defaultdict(lambda: {"count":0, "params":0})
for r in audit_rows:
    key = (r["family"], r["module_type"])
    acc[key]["count"] += 1
    acc[key]["params"] += int(r["params"])
for (fam, typ), v in sorted(acc.items()):
    summary.append({"family": fam, "module_type": typ, "count": v["count"], "params": v["params"]})
print(f"{'family':14s} {'module_type':10s} {'count':>6s} {'params(M)':>10s}")
print('-'*46)
for s in summary:
    print(f"{s['family']:14s} {s['module_type']:10s} {s['count']:6d} {s['params']/1e6:10.3f}")
print("total quantizable modules:", len(audit_rows))
print("Monarch modules:", sum(r['module_type']=='Monarch' for r in audit_rows))
print("Linear modules:", sum(r['module_type']=='Linear' for r in audit_rows))


## Section 2 — W8A8 wrapper implementation

The implementation lives in `quant_w8a8.py`. Dense `nn.Linear` weights use symmetric int8 per-output-channel quantization. Monarch `left[d,b,a]` is quantized per `(d,a)` over `b`, and `right[d,a,c]` is quantized per `(d,a)` over `c`, matching the actual einsum layout in `monarch_paper.py`. Activations default to signed symmetric dynamic per-token W8A8 quantization over the last dimension; the previous static per-module observer was too brittle for VLM hidden-state ranges.


In [ ]:
# Section 3 — Calibration data
from torch.utils.data import Dataset, DataLoader

CALIB_SAMPLES = 128
BATCH_SIZE = 2
MAX_SEQ_LEN = 1536

CACHE_JSON = (Path.home() / ".cache/huggingface/hub"
              / "datasets--liuhaotian--LLaVA-Instruct-150K"
              / "snapshots"
              / "9d451dc7629cfe0469f6ae4432b765cd603d5fcb"
              / "llava_instruct_150k.json")

dataset = None
DATA_SOURCE = None
if CACHE_JSON.exists():
    try:
        from datasets import load_dataset
        dataset = load_dataset("json", data_files=str(CACHE_JSON), split=f"train[:{CALIB_SAMPLES}]")
        DATA_SOURCE = "llava_instruct_cache"
    except Exception as e:
        print("cache load failed:", repr(e))
if dataset is None:
    try:
        from datasets import load_dataset
        dataset = load_dataset("json", data_files="hf://datasets/liuhaotian/LLaVA-Instruct-150K/llava_instruct_150k.json", split=f"train[:{CALIB_SAMPLES}]")
        DATA_SOURCE = "llava_instruct_hf"
    except Exception as e:
        print("HF load failed:", repr(e))
if dataset is None:
    DATA_SOURCE = "synthetic"
    synth = []
    for color in ["red", "blue", "green", "yellow"] * 32:
        img = Image.new("RGB", (256, 256), color)
        synth.append({"img": img, "question": "What color is this image?", "answer": color})
    dataset = synth[:CALIB_SAMPLES]
print("DATA_SOURCE:", DATA_SOURCE, "samples:", len(dataset))

class VQADataset(Dataset):
    def __init__(self, data, source):
        self.data = data
        self.source = source
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        if self.source.startswith("llava"):
            convs = item.get("conversations", [])
            q = next((c["value"] for c in convs if c.get("from") == "human"), "").replace("<image>", "").strip()
            a = next((c["value"] for c in convs if c.get("from") == "gpt"), "")
            img_data = item.get("image")
            if isinstance(img_data, str):
                try: img = Image.open(img_data).convert("RGB")
                except Exception: img = Image.new("RGB", (256, 256), "gray")
            else:
                img = Image.new("RGB", (256, 256), "gray")
        else:
            img, q, a = item["img"], item["question"], item["answer"]
        return {"image": img, "question": q or "Describe this image.", "answer": a or "unknown"}

def collate_fn(batch):
    return {"images":[b["image"] for b in batch], "questions":[b["question"] for b in batch], "answers":[b["answer"] for b in batch]}

calib_loader = DataLoader(VQADataset(dataset, DATA_SOURCE), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)
print("calibration batches:", len(calib_loader))


In [ ]:
# Shared forward helpers

def build_inputs(proc, images, questions, answers=None, device=DEVICE):
    messages = [[{"role":"user", "content":[{"type":"image"}, {"type":"text", "text": q or "Describe this image."}]}] for q in questions]
    prompts = [proc.apply_chat_template(m, add_generation_prompt=True) for m in messages]
    enc = proc(text=prompts, images=images, return_tensors="pt", padding=True)
    if answers is None:
        return enc.to(device)
    prompt_ids = enc["input_ids"]
    prompt_mask = enc.get("attention_mask", torch.ones_like(prompt_ids))
    pad_id = proc.tokenizer.pad_token_id or proc.tokenizer.eos_token_id
    rows, label_rows, keep = [], [], []
    for i, ans in enumerate(answers):
        prompt_len = int(prompt_mask[i].sum().item())
        if prompt_len >= MAX_SEQ_LEN:
            continue
        ids = prompt_ids[i, :prompt_len]
        max_ans = MAX_SEQ_LEN - prompt_len
        ans_ids = proc.tokenizer(ans, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_ans)["input_ids"][0]
        ans_ids = torch.cat([ans_ids, torch.tensor([proc.tokenizer.eos_token_id], dtype=torch.long)])[:max_ans]
        row = torch.cat([ids, ans_ids])[:MAX_SEQ_LEN]
        lab = row.clone(); lab[:min(prompt_len, lab.numel())] = -100
        rows.append(row); label_rows.append(lab); keep.append(i)
    if not rows:
        return None
    max_len = max(r.numel() for r in rows)
    input_ids = torch.full((len(rows), max_len), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((len(rows), max_len), dtype=torch.long)
    labels = torch.full((len(rows), max_len), -100, dtype=torch.long)
    for j, (row, lab) in enumerate(zip(rows, label_rows)):
        input_ids[j, :row.numel()] = row
        attention_mask[j, :row.numel()] = 1
        labels[j, :lab.numel()] = lab
    extra = {k: v[keep] for k, v in enc.items() if k not in ("input_ids", "attention_mask")}
    extra.update({"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels})
    return {k: v.to(device) for k, v in extra.items()}

@torch.no_grad()
def calibrate_model(qmodel, loader, limit_batches=None):
    qmodel.eval()
    start_calibration(qmodel)
    n = 0
    for batch in loader:
        inputs = build_inputs(processor, batch["images"], batch["questions"], device=DEVICE)
        _ = qmodel(**inputs)
        n += 1
        if limit_batches is not None and n >= limit_batches:
            break
    freeze_calibration(qmodel)
    return n

def smolvlm_query(mdl, image, question, max_new_tokens=64):
    target_device = next(mdl.parameters()).device
    messages = [{"role":"user", "content":[{"type":"image"}, {"type":"text", "text": question}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[image], return_tensors="pt").to(target_device)
    with torch.inference_mode():
        out = mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = out[:, inputs["input_ids"].shape[-1]:]
    return processor.batch_decode(gen, skip_special_tokens=True)[0].strip()


In [ ]:
# Section 4 — PTQ dynamic W8A8 quantization
print("Quantizing all Monarch and dense Linear modules")
monarch_fp = monarch_fp.to("cpu").eval()
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
monarch_w8a8_ptq, ptq_manifest = quantize_model_w8a8(monarch_fp, mode="ptq_dynamic")
monarch_w8a8_ptq = monarch_w8a8_ptq.to(DEVICE).eval()
print("quantized wrappers:", len(ptq_manifest))
print(f"{'family':14s} {'module_type':10s} {'count':>6s} {'params(M)':>10s}")
print('-'*46)
for s in manifest_summary(ptq_manifest):
    print(f"{s['family']:14s} {s['module_type']:10s} {s['count']:6d} {s['params']/1e6:10.3f}")

calib_batches = calibrate_model(monarch_w8a8_ptq, calib_loader, limit_batches=4)
print("calibration batches run (dynamic activations; stats not required):", calib_batches)


In [ ]:
# Section 4a — Weight quantization error tables
weight_errors = quant_weight_errors(monarch_fp.cpu().eval(), monarch_w8a8_ptq.cpu().eval(), ptq_manifest)
monarch_fp.to("cpu"); monarch_w8a8_ptq.to("cpu")

def print_error_table(rows, module_type):
    filt = [r for r in rows if r["module_type"] == module_type]
    fams = defaultdict(list)
    for r in filt: fams[r["family"]].append(r["weight_rel_fro_error"])
    print(f"{module_type} weight quantization error")
    print(f"{'family':14s} {'count':>6s} {'mean_rel':>10s} {'max_rel':>10s}")
    print('-'*46)
    for fam, vals in sorted(fams.items()):
        print(f"{fam:14s} {len(vals):6d} {sum(vals)/len(vals):10.6f} {max(vals):10.6f}")

print_error_table(weight_errors, "Monarch")
print()
print_error_table(weight_errors, "Linear")


In [ ]:
# Section 4b — Activation-based layer output error on representative modules

def representative_names(manifest, per_type=6):
    out=[]; seen=set()
    for typ in ["Monarch", "Linear"]:
        for r in manifest:
            key=(typ,r['family'])
            if r['module_type']==typ and key not in seen:
                out.append(r['name']); seen.add(key)
            if sum(1 for n in out if next(x for x in manifest if x['name']==n)['module_type']==typ) >= per_type:
                break
    return out

@torch.no_grad()
def activation_error_sample(fp_model, q_model, names, loader, max_batches=2):
    fp_out, q_out = {}, {}
    hooks=[]
    def store_fp(name):
        def hook(module, inputs, output):
            fp_out[name] = output.detach().float().cpu()
            return None
        return hook
    def store_q(name):
        def hook(module, inputs, output):
            q_out[name] = output.detach().float().cpu()
            return None
        return hook
    for name in names:
        hooks.append(get_module(fp_model, name).register_forward_hook(store_fp(name)))
        hooks.append(get_module(q_model, name).register_forward_hook(store_q(name)))
    fp_model.eval(); q_model.eval()
    for bi,batch in enumerate(loader):
        inputs=build_inputs(processor,batch['images'],batch['questions'],device=DEVICE)
        fp_out.clear(); q_out.clear()
        _=fp_model(**inputs)
        _=q_model(**inputs)
        break
    for h in hooks: h.remove()
    rows=[]
    for name in names:
        a=fp_out[name].reshape(-1).float(); b=q_out[name].reshape(-1).float()
        ad = a.double(); bd = b.double()
        cos = float((ad @ bd) / (ad.norm().clamp_min(1e-12) * bd.norm().clamp_min(1e-12)))
        cos = max(min(cos, 1.0), -1.0)
        rel=((a-b).norm()/a.norm().clamp_min(1e-12)).item()
        meta=next(r for r in ptq_manifest if r['name']==name)
        rows.append({**meta, 'activation_cosine': cos, 'activation_rel_error': rel})
    return rows

sample_names = representative_names(ptq_manifest)
monarch_fp = monarch_fp.to(device=DEVICE, dtype=DTYPE).eval()
monarch_w8a8_ptq = monarch_w8a8_ptq.to(device=DEVICE).eval()
act_errors = activation_error_sample(monarch_fp, monarch_w8a8_ptq, sample_names, calib_loader)
monarch_fp = monarch_fp.to("cpu").eval()
monarch_w8a8_ptq = monarch_w8a8_ptq.to("cpu").eval()
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
for typ in ["Monarch", "Linear"]:
    print(f"{typ} activation output error")
    print(f"{'family':14s} {'cosine':>10s} {'rel_error':>10s} name")
    print('-'*80)
    for r in act_errors:
        if r['module_type']==typ:
            print(f"{r['family']:14s} {r['activation_cosine']:10.6f} {r['activation_rel_error']:10.6f} {r['name']}")


In [ ]:
# Section 4c — Three-way regression QA
reg_qa = [
    {"image": Image.open(IMAGE_PATH).convert("RGB"), "question": "Describe this image in detail.", "keywords": ["cat", "animal", "fur", "eyes"]},
    {"image": Image.open(IMAGE_PATH).convert("RGB"), "question": "What is the main subject?", "keywords": ["cat"]},
    {"image": Image.open(IMAGE_PATH).convert("RGB"), "question": "What color is the cat?", "keywords": ["brown", "orange", "gray", "grey"]},
]

def run_regression(mdl, label):
    mdl = mdl.to(device=DEVICE, dtype=DTYPE).eval()
    rows=[]
    for item in reg_qa:
        ans=smolvlm_query(mdl,item['image'],item['question'])
        ok=any(k.lower() in ans.lower() for k in item['keywords'])
        rows.append({'question':item['question'], 'answer':ans, 'pass':ok, 'keywords':item['keywords']})
    score=sum(r['pass'] for r in rows)
    print(f"[{label}] {score}/{len(rows)}")
    for r in rows:
        print(('✓' if r['pass'] else '✗'), 'Q:', r['question'])
        print('  A:', r['answer'][:180])
    mdl.to("cpu")
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return {'label':label, 'score':score, 'total':len(rows), 'rows':rows}

reg_original = run_regression(original_dense, 'original dense fp')
reg_monarch = run_regression(monarch_fp, 'Monarch fp')
reg_ptq = run_regression(monarch_w8a8_ptq, 'Monarch + W8A8 PTQ')


In [ ]:
# Section 5 — QAT decision gate
CRITERION = "Run QAT if dynamic PTQ regression drops by more than 1/3 item vs Monarch-fp, or any sampled family activation cosine < 0.95."
ptq_score_drop = reg_monarch['score'] - reg_ptq['score']
min_act_cos = min(r['activation_cosine'] for r in act_errors)
NEEDS_QAT = (ptq_score_drop > 1) or (min_act_cos < 0.95)
print('criterion:', CRITERION)
print('ptq_score_drop:', ptq_score_drop)
print('min_sampled_activation_cosine:', min_act_cos)
print('NEEDS_QAT:', NEEDS_QAT)


In [ ]:
# Section 6 — QAT, only if Section 5 decided it is needed
QAT_EPOCHS = 1
QAT_MAX_BATCHES = 32
QAT_LR = 5e-5
RUN_QAT = bool(NEEDS_QAT)
print('RUN_QAT:', RUN_QAT)

qat_w8a8 = None
qat_manifest = None
reg_qat = None
qat_weight_errors = None
qat_act_errors = None

if RUN_QAT:
    print('Building QAT model with trainable FP master weights/factors plus fake W8A8 in forward')
    monarch_fp = monarch_fp.to('cpu').eval()
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    qat_model, qat_manifest = quantize_model_w8a8(monarch_fp, mode='qat')
    qat_model = qat_model.to(device=DEVICE, dtype=DTYPE).train()

    # Calibrate activation ranges for QAT fake quant.
    calibrate_model(qat_model, calib_loader, limit_batches=16)
    qat_model.train()

    for p in qat_model.parameters():
        p.requires_grad_(False)
    for _, m in iter_quant_wrappers(qat_model):
        for p in m.parameters(recurse=False):
            p.requires_grad_(True)
    trainable = sum(p.numel() for p in qat_model.parameters() if p.requires_grad)
    print('QAT trainable parameters:', trainable)

    opt = torch.optim.AdamW([p for p in qat_model.parameters() if p.requires_grad], lr=QAT_LR, weight_decay=0.0)
    step = 0
    for epoch in range(QAT_EPOCHS):
        for batch in calib_loader:
            inputs = build_inputs(processor, batch['images'], batch['questions'], batch['answers'], device=DEVICE)
            if inputs is None:
                continue
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=DTYPE, enabled=str(DEVICE).startswith('cuda')):
                out = qat_model(**inputs)
                loss = out.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in qat_model.parameters() if p.requires_grad], 1.0)
            opt.step()
            step += 1
            if step % 8 == 0:
                print('QAT step', step, 'loss', float(loss.detach().cpu()))
            if step >= QAT_MAX_BATCHES:
                break
        if step >= QAT_MAX_BATCHES:
            break
    print('QAT completed steps:', step)

    qat_model = qat_model.to('cpu').eval()
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    qat_w8a8, qat_manifest = freeze_qat_model_w8a8(qat_model)
    qat_w8a8 = qat_w8a8.to(device=DEVICE).eval()
    calibrate_model(qat_w8a8, calib_loader, limit_batches=16)
    qat_w8a8 = qat_w8a8.to('cpu').eval()

    qat_weight_errors = quant_weight_errors(monarch_fp.cpu().eval(), qat_w8a8.cpu().eval(), qat_manifest)
    qat_w8a8 = qat_w8a8.to(device=DEVICE).eval()
    monarch_fp = monarch_fp.to(device=DEVICE, dtype=DTYPE).eval()
    qat_act_errors = activation_error_sample(monarch_fp, qat_w8a8, representative_names(qat_manifest), calib_loader)
    monarch_fp = monarch_fp.to('cpu').eval(); qat_w8a8 = qat_w8a8.to('cpu').eval()
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

    print('QAT weight error summary')
    for typ in ['Monarch', 'Linear']:
        print_error_table(qat_weight_errors, typ)
    print('QAT activation error summary')
    for typ in ['Monarch', 'Linear']:
        print(f'{typ} activation output error')
        for r in qat_act_errors:
            if r['module_type'] == typ:
                print(f"{r['family']:14s} {r['activation_cosine']:10.6f} {r['activation_rel_error']:10.6f} {r['name']}")

    reg_qat = run_regression(qat_w8a8, 'Monarch + W8A8 QAT')
else:
    print('Skipping QAT because PTQ passed the stated gate.')


In [ ]:
# Section 7 — Save W8A8 checkpoint and metadata
OUT_DIR = CKPT_DIR / 'w8a8_final'
OUT_DIR.mkdir(parents=True, exist_ok=True)
if reg_qat is not None and qat_w8a8 is not None:
    chosen_model = qat_w8a8
    chosen_label = 'qat'
    chosen_manifest = qat_manifest
    chosen_weight_errors = qat_weight_errors
    chosen_activation_errors = qat_act_errors
    chosen_regression = reg_qat
else:
    chosen_model = monarch_w8a8_ptq
    chosen_label = 'ptq_dynamic'
    chosen_manifest = ptq_manifest
    chosen_weight_errors = weight_errors
    chosen_activation_errors = act_errors
    chosen_regression = reg_ptq

torch.save(chosen_model.cpu().state_dict(), OUT_DIR / 'w8a8_state_dict.pt')
processor.save_pretrained(str(OUT_DIR))
metadata = {
    'source_checkpoint': str(MONARCH_CKPT),
    'quantization': chosen_label,
    'manifest': chosen_manifest,
    'audit_summary': summary,
    'weight_errors': chosen_weight_errors,
    'activation_errors': chosen_activation_errors,
    'regression': {'original_dense': reg_original, 'monarch_fp': reg_monarch, 'ptq': reg_ptq, 'qat': reg_qat},
    'qat_needed': NEEDS_QAT,
    'note': 'RPi5 speedup has NOT been measured in this notebook; that is the next runtime stage.'
}
with open(OUT_DIR / 'w8a8_manifest.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('saved:', OUT_DIR)
print('final quantization:', chosen_label)
print('final regression score:', chosen_regression['score'], '/', chosen_regression['total'])
print('RPi5 speedup has NOT been measured here; next stage must benchmark runtime separately.')
